In [ ]:
%%capture
# 폰트 및 패키지 설치 (2~3분 소요)

# 1. 나눔 폰트 설치
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

In [ ]:
#필요 라이프러리 임포트
import pandas as pd
import requests

import re

from pathlib import Path

In [ ]:
#데이터 추출시, 문제가 있는지 확인할때 쓰이는 try-exception 클래스
class JeJuDataError(Exception):
    pass

def check_data_nums(org, ext):
    if org != ext:
        raise JeJuDataError("모든 데이터를 추출하지 못했습니다")

def check_duplicates(nums):
    if nums > 0:
        raise JeJuDataError("중복 데이터가 생겼습니다")

In [ ]:
def save_data_as_parquet(df, filename):
    df.to_parquet(f"{filename}_jeju.parquet", index=False)

    print(f"{filename} 파일 저장 완료")

def get_jeju_data(filename, columns):
    file_path = Path(f"{filename}_jeju.parquet")

    if file_path.is_file():
        print("파일 이미 존제", end=', ')
        return False

    print(f'currently reading {filename}.xlsx')                 #읽는 파일명 이름 표기

    #제주 데이터를 담은 데이터프레임 저장 리스트
    df_list =[]

    #각 반복마다 읽을 데이터 수
    chunk_size = 50000

    # 파일을 열기 => 반복할때마다 파일을 열지 않아도 됨
    try:
        with pd.ExcelFile(f'{filename}.xlsx') as xls:
            #지역 컬럼만 따로 읽기
            df_loc = pd.read_excel(xls, usecols=["지역"])

            print(f'reading rows from')     #읽고 있는 행 범위 표기
            for i in range(0, len(df_loc), chunk_size):
                print(f'{i+1} to {i+chunk_size}', end=" | ")        #읽고 있는 행 범위 표기

                #청크 사이즈 마다 읽은 데이터 => 임시 데이터프레임에 저장
                df_temp = pd.read_excel(
                    xls, skiprows=i+1, nrows=chunk_size,
                    header=None, names=columns
                )

                #'지역에 '제주'가 들어있는 데이터 행 추출
                jeju_rows = df_temp[df_temp["지역"].str.contains(r"^제주", regex=True, na=False)]
                #데이터 행 리스트에 저장
                df_list.append(jeju_rows)

        df_result = pd.concat(df_list, ignore_index=True)

    except FileNotFoundError:
        print("파일 찾을수 없음", end=", ")
        return False

    #제주관련 데이터가 추출중 문제가 있었는지 확인
    try:
        jeju_count_org = len(df_loc[df_loc["지역"].str.contains(r"^제주", regex=True, na=False)])

        check_data_nums(jeju_count_org, len(df_result))
        check_duplicates(df_result.duplicated().sum())

    except JeJuDataError as e:
        print(f'{filename} 파일에서 ', e)

    finally:
        print(f'{filename} 파일에서 제주 데이터를 추출 완료했습니다')
        return df_result

In [ ]:
files = [
    "2023_12",
    "2024_01", "2024_02", "2024_03", "2024_04", "2024_05", "2024_06", "2024_07", "2024_08", "2024_09", "2024_10", "2024_11", "2024_12",
    "2025_01", "2025_02", "2025_03", "2025_04", "2025_05", "2025_06", "2025_07", "2025_08", "2025_09", "2025_10", "2025_11", "2025_12",
    "2026_01", "2026_02", "2026_03", "2026_04", "2026_05", "2026_06"
]
columns = pd.read_excel("2026_01.xlsx", nrows=0).columns

for filename in files:
    temp = get_jeju_data(filename, columns)

    if not isinstance(temp, pd.DataFrame):
        print("전달된 변수가 데이터프레임이 아님")
    else:
        save_data_as_parquet(temp, filename)

currently reading 2023_12.xlsx
reading rows from
1 to 50000 | 50001 to 100000 | 100001 to 150000 | 150001 to 200000 | 200001 to 250000 | 250001 to 300000 | 300001 to 350000 | 350001 to 400000 | 400001 to 450000 | 450001 to 500000 | 500001 to 550000 | 550001 to 600000 | 600001 to 650000 | 650001 to 700000 | 700001 to 750000 | 2023_12 파일에서 제주 데이터를 추출 완료했습니다
2023_12 파일 저장 완료
파일 이미 존제, 전달된 변수가 데이터프레임이 아님
파일 이미 존제, 전달된 변수가 데이터프레임이 아님
파일 이미 존제, 전달된 변수가 데이터프레임이 아님
파일 이미 존제, 전달된 변수가 데이터프레임이 아님
파일 이미 존제, 전달된 변수가 데이터프레임이 아님
파일 이미 존제, 전달된 변수가 데이터프레임이 아님
파일 이미 존제, 전달된 변수가 데이터프레임이 아님
파일 이미 존제, 전달된 변수가 데이터프레임이 아님
파일 이미 존제, 전달된 변수가 데이터프레임이 아님
파일 이미 존제, 전달된 변수가 데이터프레임이 아님
파일 이미 존제, 전달된 변수가 데이터프레임이 아님
파일 이미 존제, 전달된 변수가 데이터프레임이 아님
파일 이미 존제, 전달된 변수가 데이터프레임이 아님
파일 이미 존제, 전달된 변수가 데이터프레임이 아님
파일 이미 존제, 전달된 변수가 데이터프레임이 아님
파일 이미 존제, 전달된 변수가 데이터프레임이 아님
파일 이미 존제, 전달된 변수가 데이터프레임이 아님
파일 이미 존제, 전달된 변수가 데이터프레임이 아님
파일 이미 존제, 전달된 변수가 데이터프레임이 아님
파일 이미 존제, 전달된 변수가 데이터프레임이 아님
파일 이미 존제, 전달된 변수가 데이터프레임이 아님
파일 이미 존제, 전달된 변수

In [ ]:
df_example = pd.read_parquet('2024_01_jeju.parquet')

df_example.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34049 entries, 0 to 34048
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   충전소명         34049 non-null  object 
 1   충전기ID        34049 non-null  int64  
 2   충전소 유형(대분류)  34049 non-null  object 
 3   충전소 유형(소분류)  34049 non-null  object 
 4   지역           34049 non-null  object 
 5   시군구          34049 non-null  object 
 6   주소           34049 non-null  object 
 7   충전기타입        34049 non-null  object 
 8   충전기용량(KW)    34049 non-null  object 
 9   충전시작일시       34049 non-null  int64  
 10  충전종료일시       34049 non-null  int64  
 11  충전시간         34049 non-null  object 
 12  충전량          34049 non-null  float64
dtypes: float64(1), int64(3), object(9)
memory usage: 3.4+ MB


# 모든 데이터 불러오기 (년도_월 별로 정리)

In [ ]:
files = [
    "2023_12",
    "2024_01", "2024_02", "2024_03", "2024_04", "2024_05", "2024_06", "2024_07", "2024_08", "2024_09", "2024_10", "2024_11", "2024_12",
    "2025_01", "2025_02", "2025_03", "2025_04", "2025_05", "2025_06", "2025_07", "2025_08", "2025_09", "2025_10", "2025_11", "2025_12",
    "2026_01", "2026_02", "2026_03", "2026_04", "2026_05", "2026_06"
]

#모든 제주데이터 읽기
dfs = [pd.read_parquet(f'{file}_jeju.parquet') for file in files]

# 데이터 더 알아보기
카테고리 형 데이터에 어떤 카테고리들잉 있는지 분석하여 시간 변화에 따라 바뀐 카테고리가 있는지 확인

In [ ]:
#제주 데이터 파일들에서 컬럼의 카테고리 추출
region_type_large_category = [set(df["충전소 유형(대분류)"].dropna()) for df in dfs]
region_type_small_category = [set(df["충전소 유형(소분류)"].dropna()) for df in dfs]
charger_type_category = [set(df["충전기타입"].dropna()) for df in dfs]
charger_capacity_category = [set(df["충전기용량(KW)"].dropna()) for df in dfs]

#데이터 파일들의 컬럼 비교 (모두 같으면 True, 아닌면 False)
all_same_region_large = all(s == region_type_large_category[0] for s in region_type_large_category)
all_same_region_small = all(s == region_type_small_category[0] for s in region_type_small_category)
all_same_charger_type = all(s == charger_type_category[0] for s in charger_type_category)
all_same_charger_capacity = all(s == charger_capacity_category[0] for s in charger_capacity_category)

print(all_same_region_large)
print(all_same_region_small)
print(all_same_charger_type)
print(all_same_charger_capacity)

True
True
True
False


In [ ]:
# 카테고리가 같지 않던 "충전기용량(KW)"컬럼 확인 => 어떤 파일에서 같지 않았던 건지 확인 및 출력
expected = charger_capacity_category[0]

for i, s in enumerate(charger_capacity_category):
    print("Dataset", i)

    print("Missing:", expected - s)
    print("Extra:", s - expected)

Dataset 0
Missing: set()
Extra: set()
Dataset 1
Missing: set()
Extra: set()
Dataset 2
Missing: set()
Extra: set()
Dataset 3
Missing: set()
Extra: set()
Dataset 4
Missing: set()
Extra: set()
Dataset 5
Missing: set()
Extra: set()
Dataset 6
Missing: set()
Extra: set()
Dataset 7
Missing: set()
Extra: set()
Dataset 8
Missing: set()
Extra: set()
Dataset 9
Missing: set()
Extra: set()
Dataset 10
Missing: set()
Extra: set()
Dataset 11
Missing: set()
Extra: set()
Dataset 12
Missing: set()
Extra: set()
Dataset 13
Missing: set()
Extra: set()
Dataset 14
Missing: set()
Extra: set()
Dataset 15
Missing: set()
Extra: set()
Dataset 16
Missing: set()
Extra: set()
Dataset 17
Missing: set()
Extra: set()
Dataset 18
Missing: set()
Extra: set()
Dataset 19
Missing: set()
Extra: set()
Dataset 20
Missing: set()
Extra: set()
Dataset 21
Missing: set()
Extra: set()
Dataset 22
Missing: set()
Extra: set()
Dataset 23
Missing: set()
Extra: set()
Dataset 24
Missing: set()
Extra: set()
Dataset 25
Missing: set()
Extra: se

In [ ]:
print(region_type_large_category[0])
print(region_type_small_category[0])
print(charger_type_category[0])
print(charger_capacity_category[0])

{'휴게시설', '상업시설', '주차시설', '관광시설', '근린생활시설', '공공시설', '기타시설', '공동주택시설', '교육문화시설'}
{'경찰서', '전시관', '주유소', '지방도로 휴게소', '박물관', '공원주차장', '공공기관', '교육원', '마트(쇼핑몰)', '관공서', '주민센터', '공원', '지자체시설', '경기장', '기타', '보건소', '학교', '홍보관', '음식점', '야영장', '일반주차장', '민속마을', '수련원', '관광안내소', '유적지', '도서관', '관광지', '숙박시설', '공영주차장', '아파트', '공연장', '복지관'}
{'DC콤보', 'DC차데모+AC3상+DC콤보'}
{'급속(300kW동시)', '급속(100kW동시)', '급속(200kW동시)', '급속(300kW단독)', '급속(400kW동시)', '급속(100kW단독)', '급속(50kW)', '급속(100kW멀티)'}


카테고리 형식 컬럼의 카데고리 목록
- 충전소 유형(대분류)<br>
{'휴게시설', '상업시설', '주차시설', '관광시설', '근린생활시설', '공공시설', '기타시설', '공동주택시설', '교육문화시설'}

- 충전소 유형(대분류) <br>
{'경찰서', '전시관', '주유소', '지방도로 휴게소', '박물관', '공원주차장', '공공기관', '교육원', '마트(쇼핑몰)', '관공서', '주민센터', '공원', '지자체시설', '경기장', '기타', '보건소', '학교', '홍보관', '음식점', '야영장', '일반주차장', '민속마을', '수련원', '관광안내소', '유적지', '도서관', '관광지', '숙박시설', '공영주차장', '아파트', '공연장', '복지관'}

- 충전기 타입<br>
{'DC콤보', 'DC차데모+AC3상+DC콤보'}

- 충전기 용량<br>
{'급속(300kW동시)', '급속(100kW동시)', '급속(200kW동시)', '급속(300kW단독)', '급속(400kW동시)', '급속(100kW단독)', '급속(50kW)', '급속(100kW멀티)'}

# 데이터 전처리
- 년도 별로 합치기
    - 컬럼 이름 정리 (반복되는 충전기/충전 제거)
    - 충전시작일시, 충전종료일시 순으로 재정리

<br>

- 필요없는 컬럼 삭제
    - 지역 : 모든 지역은 '제주특별자치도'
    - 충전량 : 충전량 요소는 사용시간/충전기용량에 비래하기에 혼잡도 예상도에 대한 요소로 비적합

<br>

- 주소 : 반복되는 주소 삭제 (.duplicated 로는 발견 X)
- 충전리 용량: '급속' 이라는 단어는 모든 데이터에 반복 (의미 X)

In [ ]:
def concat_by_year(filenames, year):
    #달별로 읽기
    dfs_year = [pd.read_parquet(f'{file}_jeju.parquet') for file in filenames]

    #달별로 있는 데이터셋 합치기
    df_year = (
        pd.concat([df for df in dfs_year], ignore_index=True)
        .sort_values(by=["충전시작일시", "충전종료일시"])
        .reset_index(drop=True)
    )

    #필요없는 컬럼 삭제
    df_year.drop(columns=["지역", "충전량"], inplace=True)

    # 새로운 컬럼 이름
    new_columns = [
        "충전소명", "충전기ID", "유형(대분류)", "유형(소분류)", "시군구", "주소",
        "타입", "용량(KW)", "시작일시", "종료일시", "충전시간"
    ]
    df_year.columns = new_columns

    #parquet 포멧으로 저장
    df_year.to_parquet(f"{year}_jeju.parquet")

In [ ]:
years = [2023, 2024, 2025, 2026]
files = [
    ["2023_12"],
    ["2024_01", "2024_02", "2024_03", "2024_04", "2024_05", "2024_06", "2024_07", "2024_08", "2024_09", "2024_10", "2024_11", "2024_12"],
    ["2025_01", "2025_02", "2025_03", "2025_04", "2025_05", "2025_06", "2025_07", "2025_08", "2025_09", "2025_10", "2025_11", "2025_12"],
    ["2026_01", "2026_02", "2026_03", "2026_04", "2026_05", "2026_06"]
]

for i, year in enumerate(years):
    concat_by_year(files[i], year)

In [ ]:
#년도 별로 데이터 읽기
years = [2023, 2024, 2025, 2026]

dfs_dirty = [pd.read_parquet(f'{year}_jeju.parquet') for year in years]

In [ ]:
#주소 컬럼 전처리
dfs_address = [df.copy(deep=True) for df in dfs_dirty]

#반복되는 주소 삭제
for df in dfs_address:
    df["주소"] = (
        df["주소"].str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .str.replace(
            r"^(.+?)\s+\1(?:\s+.*)?$",
            r"\1",
            regex=True
        )
    )

In [ ]:
#충전기 용량 컬럼 전처리
dfs_capacity = [df.copy(deep=True) for df in dfs_address]

#의미 없는 단어 요소에서 삭제
for df in dfs_capacity:
    df["용량(KW)"] = (
        df["용량(KW)"]
        .str.replace("급속(", "", regex=False)
        .str.replace(")", "", regex=False)
    )

In [ ]:
for i, df in enumerate(dfs_capacity):
  df.to_parquet(f"{years[i]}_jeju_cleaned.parquet")

## 서비스기간 사용할 충전소 데이터베이스 (표준화)

(기본키) - 충전소명 - 충전기ID - 충전소유형(대, 소) - 시군구 - 주소 - 충전기타입 - 충전기용량

<br>

## 훈련 데이터 요소 (X 데이터)

### 충전기 위치/유형 요소
충전소ID(외례키) - 충전소유형(대) - 충전소유형(소) - 충전기타입 - 충전기용량 - 위도 - 경도

### 기간
시 - 일

### 시간적 요소
(지난 시간 사용량) <br>
usage_lag_1h - usage_lag_2h - usage_lag_3h - usage_lag_24h - usage_lag_168h

(지난 시간 평균 사용량) <br>
rolling_mean_3h - rolling_mean_24h

<br>

## 검증 요소 (y 데이터)
당시 시간의 사용량

In [ ]:
#년도 별로 데이터 읽기
years = [2023, 2024, 2025, 2026]

dfs_nf0 = [pd.read_parquet(f'{year}_jeju_cleaned.parquet') for year in years]

# NF1
- 용량(KW) => 용량(KW) + 모드 분리
- 충전시작/종료일시 => datetime 데이터 타입으로 변환
    - 시간별 사용빈도수를 계산하기 위해서 분리하지 않는게 더 유리

In [ ]:
#전처리전 데이터 복사
dfs_nf1 = [df.copy(deep=True) for df in dfs_nf0]

In [ ]:
#충전기 용량 컬럼 전처리

#용량(KW) 컬럼 표준화
for df in dfs_nf1:
    #용량(KW) + 모드로 분리
    df[["용량_kw", "모드"]] = df["용량(KW)"].str.extract(
        r"(\d+(?:\.\d+)?)\s*kW\s*(.*)"
    )

    #용량의 dtype을 숫자로 전환
    df["용량_kw"] = pd.to_numeric(df["용량_kw"])
    #모드의 NA를 ""로 전환(의미있는 데이터라는 표시)
    df["모드"] = df["모드"].replace("", pd.NA)

    df.drop(columns=["용량(KW)"], inplace=True)

In [ ]:
#충전 시작/종료 일시 컬럼 전처리 (데이터 타입)
for df in dfs_nf1:
    for col in ["시작", "종료"]:
        df[f"{col}일시"] = pd.to_datetime(
            df[f"{col}일시"].astype(str),
            format="%Y%m%d%H%M%S",
            errors="coerce"
        )

In [ ]:
dfs_nf1[1].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 358492 entries, 0 to 358491
Data columns (total 12 columns):
 #   Column   Non-Null Count   Dtype         
---  ------   --------------   -----         
 0   충전소명     358492 non-null  object        
 1   충전기ID    358492 non-null  int64         
 2   유형(대분류)  358492 non-null  object        
 3   유형(소분류)  358492 non-null  object        
 4   시군구      358492 non-null  object        
 5   주소       358492 non-null  object        
 6   타입       358492 non-null  object        
 7   시작일시     358492 non-null  datetime64[ns]
 8   종료일시     358492 non-null  datetime64[ns]
 9   충전시간     358492 non-null  object        
 10  용량_kw    358492 non-null  int64         
 11  모드       305855 non-null  object        
dtypes: datetime64[ns](2), int64(2), object(8)
memory usage: 32.8+ MB


# NF2
- 충전소 데이터베이스 분리
- 분리된 충전소 데이터베이스를 활용하여<br>
    - 충전소명 / 충전소id / 주소 => charger_pk 로 변경

In [ ]:
#전처리전 데이터 복사
dfs_nf2 = [df.copy(deep=True) for df in dfs_nf1]

In [ ]:
#충전소 데이터베이스 컬럼
charger_cols = ["충전소명", "충전기ID", "유형(대분류)", "유형(소분류)", "시군구", "주소", "타입", "용량_kw", "모드"]

df_chargers = (
    pd.concat([df[charger_cols] for df in dfs_nf2], ignore_index=True)    #모든 데이터 합치기
    .drop_duplicates(subset=["충전소명", "충전기ID"])                       #반복된 충전소명+충전기ID 데이터 삭제
    .sort_values(by=["충전소명", "충전기ID"])                               #충전소명->충전기ID 순으로 정리
    .reset_index(drop=True)
)

#primary key 생성
df_chargers["charger_pk"] = df_chargers.index + 1
#순서 설정
col = df_chargers.pop("charger_pk")
df_chargers.insert(0, "charger_pk", col)

#충전소 저장
df_chargers.to_parquet("chargers_jeju.parquet")
df_chargers.to_excel("chargers_jeju.xlsx")

In [ ]:
#충전기의 고유식별을 담당하는 요소들을 하나로 합침
# 충전소명 / 충전소ID / 주소 => charger_pk (primary key)
lookup = df_chargers[
    ["charger_pk", "충전소명", "충전기ID"]
]

for i, df in enumerate(dfs_nf2):
    df = df.merge(lookup, on=["충전소명", "충전기ID"], how="left")

    df.drop(columns=["충전소명", "충전기ID", "주소"], inplace=True)

    dfs_nf2[i] = df

In [ ]:
dfs_nf2[0].head()

,유형(대분류),유형(소분류),시군구,타입,시작일시,종료일시,충전시간,용량_kw,모드,charger_pk
0,주차시설,공영주차장,제주시,DC콤보,2023-12-01 00:06:58,2023-12-01 00:38:09,00:31:11,100,단독,175
1,관광시설,박물관,제주시,DC차데모+AC3상+DC콤보,2023-12-01 00:11:32,2023-12-01 00:51:38,00:40:06,100,멀티,71
2,공공시설,주민센터,제주시,DC콤보,2023-12-01 00:12:28,2023-12-01 00:21:42,00:08:37,200,동시,166
3,공공시설,지자체시설,제주시,DC콤보,2023-12-01 00:13:44,2023-12-01 00:20:22,00:06:43,100,단독,253
4,관광시설,관광지,제주시,DC차데모+AC3상+DC콤보,2023-12-01 00:16:52,2023-12-01 00:30:26,00:10:30,100,멀티,337


In [ ]:
years = [2023, 2024, 2025, 2026]

for i, df in enumerate(dfs_nf2):
  df.to_parquet(f"{years[i]}_jeju_nf.parquet")

# 파생 요소 엔지니어링
- 충전소 데이터베이스
    - API 를 활용한 주소 => 위도, 경도
    - X 데이터에 위도, 경도 추가

- X 데이터
    - 시작일시, 종료일시를 활용하여 아래 요소 생성
        - 날짜
        - 시간 (1시간 단위)
        - 시간별 사용빈도수 => y 데이터로 활용
    
    - 생선된 y-데이터를 사용하여
    
    (지난 시간 사용량)

        - usage_lag_1h (1시간전 사용빈도수)
        - usage_lag_2h (2시간전 사용빈도수)
        - usage_lag_3h (3시간전 사용빈도수)
        - usage_lag_24h (하루전 사용빈도수)
        - usage_lag_168h (1주일전 사용빈도수)

    (지난 시간 평균 사용량)
        
        - rolling_mean_3h  (지난 3시간동안 평균 사용량)
        - rolling_mean_24h (지난 하루동안 평균 사용량)

In [ ]:
def get_coordinates(address):
    #네이버 멥 API id, key
    client_id = "2b4uw6j1w3"
    client_key = "JaKcqeI9UGumxI1lM1htbaybKuOyY8kTFg9xa7Iz"

    #request get url
    url = "https://maps.apigw.ntruss.com/map-geocode/v2/geocode"

    #헤더
    headers = {
        "x-ncp-apigw-api-key-id": client_id,
        "x-ncp-apigw-api-key": client_key,
        "Accept": "application/json"
    }
    #실제 주소 입력
    params = {
        "query": address
    }

    try:
        #get request 전달
        response = requests.get(url, headers=headers, params=params)

        #데이터 추출
        data = response.json()

        #데이터가 없으면 Null value 로 대체
        if not data["addresses"]:
            return None, None

        result = data["addresses"][0]

        longitude = float(result["x"])
        latitude = float(result["y"])

        return longitude, latitude

    #get request 중 에러가 생길경우
    except requests.exceptions.HTTPError as e:
        print(f"HTTP error for {address}: {e}")
        print(response.text)
        return None, None

    except requests.exceptions.Timeout:
        print(f"Timeout: {address}")
        return None, None

    except requests.exceptions.RequestException as e:
        print(f"Request failed for {address}: {e}")
        return None, None

In [ ]:
df_chargers = pd.read_parquet('chargers_jeju.parquet')

df_chargers[["경도", "위도"]] = df_chargers["주소"].apply(
    lambda address: pd.Series(get_coordinates(address))
)

In [ ]:
df_chargers[["경도", "위도"]].isna().sum()

,0
경도,8
위도,8


In [ ]:
df_chargers[df_chargers["경도"].isna()]

,charger_pk,충전소명,충전기ID,유형(대분류),유형(소분류),시군구,주소,타입,용량_kw,모드,경도,위도
38,39,대정하모3공영주차장,1,주차시설,공영주차장,서귀포시,제주특별자치도 서귀포시 대정읍 대정읍 하모리 1513,DC차데모+AC3상+DC콤보,100,멀티,NaN,NaN
51,52,돌문화공원관리소,1,공공시설,지자체시설,제주시,제주특별자치도 제주시 조천읍 교래리 남조로 2023,DC콤보,100,단독,NaN,NaN
52,53,돌문화공원관리소,2,공공시설,지자체시설,제주시,제주특별자치도 제주시 조천읍 교래리 남조로 2023,DC콤보,100,단독,NaN,NaN
53,54,돌문화공원관리소,3,공공시설,지자체시설,제주시,제주특별자치도 제주시 조천읍 교래리 남조로 2023,DC콤보,100,단독,NaN,NaN
54,55,돌문화공원관리소,4,공공시설,지자체시설,제주시,제주특별자치도 제주시 조천읍 교래리 남조로 2023,DC콤보,100,단독,NaN,NaN
133,134,성읍119지역센터,1,공공시설,지자체시설,서귀포시,제주특별자치도 서귀포시 성산읍 성읍서문로 75,DC콤보,100,단독,NaN,NaN
157,158,안덕면사무소,11,공공시설,주민센터,서귀포시,제주특별자치도 서귀포시 안덕면 화순서로 74,DC콤보,200,동시,NaN,NaN
158,159,안덕면사무소,12,공공시설,주민센터,서귀포시,제주특별자치도 서귀포시 안덕면 화순서로 74,DC콤보,200,동시,NaN,NaN


In [ ]:
#row 38번
address = df_chargers.loc[38, "주소"]

#반복되는 단어 삭제
address = re.sub(r"\b(\S+)(?:\s+\1)+\b",r"\1",address)
address = re.sub(r"\s+", " ", address).strip()

df_chargers.at[38, "주소"] = address

df_chargers.loc[38, ["경도", "위도"]] = get_coordinates(address)

In [ ]:
df_chargers.loc[38, ["경도", "위도"]]

,38
경도,126.252534
위도,33.223891


In [ ]:
df_coor_null = df_chargers[df_chargers["경도"].isna()]

addresses = df_coor_null["주소"].unique()
coordinates = [[126.65348, 33.41806], [126.794300, 33.383441], [126.330668, 33.257492]]
coor_dict = dict(zip(addresses, coordinates))

mask = df_chargers["주소"].isin(coor_dict)

df_chargers.loc[mask, "경도"] = df_chargers.loc[mask, "주소"].map(
    lambda x: coor_dict[x][0]
)
df_chargers.loc[mask, "위도"] = df_chargers.loc[mask, "주소"].map(
    lambda x: coor_dict[x][1]
)

In [ ]:
df_chargers[["경도", "위도"]].isna().sum()

,0
경도,0
위도,0


In [ ]:
df_chargers.to_parquet("chargers_jeju_cleaned.parquet")

## X 데이터 요소
1. X 데이터 요소에 필요없는 데이터 임시 삭제 (데이터 제구성으로 인해 데이터 행수가 달라짐)
2. 사용시간 별 사용빈도 요소 생성 (사용시간에 사용빈도가 없으면 => 0 으로 설정)
3.  요소들을 charger_pk 를 활용하여 추가

### 1. feature engineering에 필요없는 데이터 삭제

In [ ]:
#필요 데이터 읽기
years = [2023, 2024, 2025, 2026]

dfs_data = [pd.read_parquet(f'{year}_jeju_nf.parquet') for year in years]

In [ ]:
#전처리전 데이터 복사
dfs_removed = [df.copy(deep=True) for df in dfs_data]

for i, df in enumerate(dfs_removed):
    dfs_removed[i] = df[['charger_pk', "시작일시", "종료일시"]]

dfs_removed[0].head()

,charger_pk,시작일시,종료일시
0,175,2023-12-01 00:06:58,2023-12-01 00:38:09
1,71,2023-12-01 00:11:32,2023-12-01 00:51:38
2,166,2023-12-01 00:12:28,2023-12-01 00:21:42
3,253,2023-12-01 00:13:44,2023-12-01 00:20:22
4,337,2023-12-01 00:16:52,2023-12-01 00:30:26


### 2. 사용기간별 사용빈도 요소 생성 (데이터셋 재구성)

In [ ]:
#전처리전 데이터 복사
dfs_usage = [df.copy(deep=True) for df in dfs_removed]

In [ ]:
def get_used_hours(row):
    start = row["시작일시"]
    end = row["종료일시"]

    if pd.isna(start) or pd.isna(end):
        return []

    # 각 시간을 hh:00 ~ hh:59로 세팅 (hh:00 ~ hh+1:00 면 겹침)
    end = end - pd.Timedelta(nanoseconds=1)

    return pd.date_range(
        start=start.floor("h"),
        end=end.floor("h"),
        freq="h"
    )

In [ ]:
for i, df in enumerate(dfs_usage):
    df["사용시간"] = df.apply(get_used_hours,axis=1)

    hourly = df.explode("사용시간")

    dfs_usage[i] = (
        hourly
        .groupby(["charger_pk", "사용시간"])
        .size()
        .reset_index(name="사용빈도수")
    )

In [ ]:
dfs_usage[0].head()

,charger_pk,사용시간,사용빈도수
0,1,2023-12-02 12:00:00,1
1,1,2023-12-02 13:00:00,1
2,1,2023-12-03 15:00:00,1
3,1,2023-12-03 20:00:00,1
4,1,2023-12-08 18:00:00,1


In [ ]:
dfs_usage[2].tail()

,charger_pk,사용시간,사용빈도수
469564,349,2025-12-31 19:00:00,1
469565,349,2025-12-31 20:00:00,2
469566,349,2025-12-31 21:00:00,1
469567,349,2025-12-31 23:00:00,1
469568,349,2026-01-01 00:00:00,1


In [ ]:
#전처리전 데이터 복사
dfs_fillna = [df.copy(deep=True) for df in dfs_usage]

In [ ]:
for i, df in enumerate(dfs_fillna):

    hours = pd.date_range(
        start = df["사용시간"].min().floor("D"),
        end = df["사용시간"].max().ceil("D") - pd.Timedelta(hours=25),
        freq="h"
    )

    # 모든 충전소 id
    stations = df["charger_pk"].unique()

    # 모든 충전소마다 각 사용시간 행 만들기 (station × hour)
    full_index = pd.MultiIndex.from_product([stations, hours], names=["charger_pk", "사용시간"])

    # 만약 사용시간에 사용수가 없으면 0으로 채움
    dfs_fillna[i] = (
        df.set_index(["charger_pk", "사용시간"])
        .reindex(full_index, fill_value=0)
        .reset_index()
    )

### 3. charger_pk를 활용하여 충전기 정보관련 요소 다시 추가

In [ ]:
#전처리전 데이터 복사
df_chargers_info = [df.copy(deep=True) for df in dfs_fillna]

df_chargers = pd.read_parquet(f'chargers_jeju_cleaned.parquet')

In [ ]:
cols_to_add = ["charger_pk", "유형(대분류)", "유형(소분류)", "시군구", "타입", "용량_kw", "모드", "경도", "위도"]

for i, df in enumerate(df_chargers_info):
    df_chargers_info[i] = df_chargers_info[i].merge(
        df_chargers[cols_to_add],
        on="charger_pk",
        how="left"
    )

In [ ]:
years = [2023, 2024, 2025, 2026]

for i, df in enumerate(df_chargers_info):
  df.to_parquet(f"{years[i]}_jeju_fe.parquet")

## train-test 데이터세트 만들기
- train : 2024-01 ~ 2025-12 (마지막 1주 제외)
- test : 2026-01 ~ 2026-06

1. train / test 끼리 합치기 (참조할 데이터 포함)
2. usage_lag + rolling mean 요소 추가
3. 참조에만 쓰이는 데이터 삭제

In [ ]:
#필요 데이터 읽기
years = [2023, 2024, 2025, 2026]

dfs_data = [pd.read_parquet(f'{year}_jeju_fe.parquet') for year in years]

In [ ]:
def split_dataset(dfs, start_date, end_date):
    temp = pd.concat(dfs, ignore_index=True)

    temp = temp[
        (temp["사용시간"] >= start_date) &
        (temp["사용시간"] < end_date)
    ].copy()

    return temp

In [ ]:
dfs_train = dfs_data[:3]
dfs_test = dfs_data[-2:]

train_set = split_dataset(dfs_train,"2023-12-25", "2026-01-01")
test_set = split_dataset(dfs_test, "2025-12-25", "2026-07-01")

In [ ]:
def usage_lag(df):
    df = df.sort_values(
        ["charger_pk", "사용시간"]
    ).reset_index(drop=True)

    df["usage_lag_1h"] = (
        df.groupby("charger_pk")["사용빈도수"]
        .shift(1)
    )

    df["usage_lag_2h"] = (
        df.groupby("charger_pk")["사용빈도수"]
        .shift(2)
    )

    df["usage_lag_24h"] = (
        df.groupby("charger_pk")["사용빈도수"]
        .shift(24)
    )

    df["usage_lag_168h"] = (
        df.groupby("charger_pk")["사용빈도수"]
        .shift(168)
    )

    return df

In [ ]:
df_train = usage_lag(train_set)

df_train = df_train[
    df_train["사용시간"] >= pd.Timestamp("2024-01-01")
].reset_index(drop=True)

df_train = df_train.sort_values(["사용시간"]).reset_index(drop=True)

df_train.to_parquet("train_jeju.parquet")

In [ ]:
df_test = usage_lag(test_set)

df_test = df_test[
    df_test["사용시간"] >= pd.Timestamp("2026-01-01")
].reset_index(drop=True)

df_test = df_test.sort_values(["사용시간"]).reset_index(drop=True)

df_test.to_parquet("test_jeju.parquet")

In [ ]:
df_test.head()

,charger_pk,사용시간,사용빈도수,유형(대분류),유형(소분류),시군구,타입,용량_kw,모드,경도,위도,usage_lag_1h,usage_lag_2h,usage_lag_24h,usage_lag_168h
0,1,2026-01-01 00:00:00,0,상업시설,음식점,제주시,DC차데모+AC3상+DC콤보,50,None,126.669703,33.536849,0.0,0.0,0.0,NaN
1,1,2026-01-01 01:00:00,0,상업시설,음식점,제주시,DC차데모+AC3상+DC콤보,50,None,126.669703,33.536849,0.0,0.0,0.0,NaN
2,1,2026-01-01 02:00:00,0,상업시설,음식점,제주시,DC차데모+AC3상+DC콤보,50,None,126.669703,33.536849,0.0,0.0,0.0,NaN
3,1,2026-01-01 03:00:00,0,상업시설,음식점,제주시,DC차데모+AC3상+DC콤보,50,None,126.669703,33.536849,0.0,0.0,0.0,NaN
4,1,2026-01-01 04:00:00,0,상업시설,음식점,제주시,DC차데모+AC3상+DC콤보,50,None,126.669703,33.536849,0.0,0.0,0.0,NaN
